# 05 — Time Series Forecasting with ARIMA (PM2.5)
Mục tiêu:
- Làm sạch + chuẩn hoá chuỗi theo tần suất giờ (hourly) và xử lý missing.
- Kiểm tra **trend / seasonality / stationarity** (ADF, KPSS), ACF/PACF.
- Chọn tham số **(p,d,q)** bằng grid nhỏ (AIC/BIC) và dự báo bằng **ARIMA**.

> Lưu ý: Notebook này **chỉ dùng ARIMA** (statsmodels). 


In [ ]:
# [Cell 1]
import sys
import os
from pathlib import Path

# Lấy đường dẫn tuyệt đối của thư mục hiện tại (notebooks/)
current_dir = Path(os.getcwd())

# Tìm thư mục gốc của dự án (nơi chứa folder src và notebooks)
# Logic: Đi ngược lên 1 cấp từ thư mục notebooks
project_root = current_dir.parent

# Thêm thư mục gốc vào sys.path để Python nhìn thấy folder 'src'
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Thử import để kiểm tra ngay lập tức
try:
    from src.classification_library import Paths
    print(f"✅ Đã import thành công src từ: {project_root}")
except ImportError as e:
    print(f"❌ Vẫn lỗi import: {e}")
    # Fallback cho trường hợp chạy local khác cấu trúc
    sys.path.append(os.path.abspath(".."))
# Parameters
RAW_ZIP_PATH = "../data/raw/PRSA2017_Data_20130301-20170228.zip"
STATION = "Aotizhongxin"
VALUE_COL = "PM2.5"
CUTOFF = "2017-01-01"
P_MAX = 3
Q_MAX = 3
D_MAX = 2
IC = "aic"
ARTIFACTS_PREFIX = "arima_pm25"

# [Cell 2]
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath(".."))
from src.classification_library import Paths
from src.timeseries_library import forecast_workflow

# Thiết lập đường dẫn
paths = Paths(project_root=Path(".."))

# [Cell 3]
# 1. CHẠY FORECAST WORKFLOW
# (Tự động Load -> Clean -> Grid Search -> Train -> Forecast)
results = forecast_workflow(
    paths=paths,
    station=STATION,
    cutoff=CUTOFF,
    p_max=P_MAX,
    q_max=Q_MAX,
    ic=IC
)

# [Cell 4]
# 2. VẼ BIỂU ĐỒ KẾT QUẢ (Forecast vs Actual)
pred_df = results["predictions"]

plt.figure(figsize=(15, 6))
# Vẽ đường thực tế
plt.plot(pred_df['datetime'], pred_df['y_true'], label="Actual (Thực tế)", color='black', alpha=0.6, linewidth=1)
# Vẽ đường dự báo
plt.plot(pred_df['datetime'], pred_df['y_pred'], label="ARIMA Forecast", color='red', linewidth=2)
# Vẽ khoảng tin cậy
plt.fill_between(pred_df['datetime'], pred_df['lower_ci'], pred_df['upper_ci'], color='red', alpha=0.1, label='95% Confidence Interval')

plt.title(f"Kết quả dự báo PM2.5 tại {STATION} (ARIMA {results['summary']['best_order']})")
plt.xlabel("Thời gian")
plt.ylabel("PM2.5")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Hiển thị metrics
print("Evaluation Metrics:")
print(results["summary"]["metrics"])